In [1]:
import glob
import warnings
from time import perf_counter

import h5py
import jax.numpy as jnp
import numpy as np
from scipy.optimize import OptimizeWarning

## Function definitions

In [2]:
from dev import single_exp, double_exp, bright, nonlinear_fit
from dev import single_exp_old, double_exp_old, bright_old, TukeyBiweight

#### load data 

In [3]:
def get_valid_corrected_f(path):
    dr = path.split("/")[-1]
    data = []
    with h5py.File(f"{path}/{dr}_data.h5") as f:
        for k in f['planes'].keys():
            data.append(f[f'planes/{k}/corrected_f'][f[f'planes/{k}/valid_roi_inds'][:]])
    return data

## Data 

In [4]:
dirs = glob.glob("/data/test-dataset-for-dff_multiplane-ophys_02/*/*")
dirs

['/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/804670',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/775682',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/782149',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753562',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/729088',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/758265',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753561',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/724567',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/755212',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/759075',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/726433',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/747443',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/757436',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/69

In [5]:
traces = get_valid_corrected_f(dirs[0])

In [6]:
[t.shape for t in traces]

[(56, 48374),
 (46, 48374),
 (73, 48374),
 (73, 48374),
 (62, 48374),
 (56, 48374),
 (41, 48374),
 (49, 48374),
 (59, 48403),
 (46, 48403),
 (71, 48403),
 (75, 48403),
 (65, 48403),
 (57, 48403),
 (39, 48403),
 (53, 48403),
 (57, 48427),
 (48, 48427),
 (67, 48427),
 (73, 48427),
 (65, 48427),
 (61, 48427),
 (41, 48427),
 (48, 48427),
 (55, 48390),
 (46, 48390),
 (65, 48390),
 (76, 48390),
 (63, 48390),
 (56, 48390),
 (40, 48390),
 (48, 48390),
 (60, 48373),
 (42, 48373),
 (65, 48373),
 (76, 48373),
 (66, 48373),
 (58, 48373),
 (40, 48373),
 (52, 48373),
 (58, 48366),
 (44, 48366),
 (66, 48366),
 (75, 48366),
 (61, 48366),
 (54, 48366),
 (42, 48366),
 (51, 48366)]

In [7]:
trace = traces[0][0]

In [8]:
frame_rate = 10.63  # looked up manually from session.json of multiplane-ophys_804670_2025-09-24_09-30-56_processed_2025-10-10_22-28-44
timestamps = np.arange(len(trace)) / frame_rate

## bounded robust regression (Tukey) 

In [9]:
single_exp_bounds = [(0, None)] * 2 + [(300, None)]
double_exp_bounds = [(0, None)] * 3 + [(300, None), (1, 1200)]
bright_bounds = [(0, None)] * 5 + [(300, None), (1, 1200), (1, 180), (60, None)]

#### old parametrization w/ multiplicative parameter interactions

In [10]:
single_exp_init = [trace[-1000:].mean(), 0.35, 3600]
double_exp_init = [trace[-1000:].mean(), 0.35, 0.2, 3600, 240]
bright_init = [trace[-1000:].mean(), 0.35, 0.2, 0.1, 0.1, 3600, 240, 50, 2000]

for model, start_params, bounds in (
    (single_exp_old, single_exp_init, single_exp_bounds),
    (double_exp_old, double_exp_init, double_exp_bounds),
    (bright_old, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for optimizer in ("Nelder-Mead", "L-BFGS-B"):
        tic = -perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = nonlinear_fit(trace, timestamps, model, start_params, bounds=bounds,
                                         M=TukeyBiweight(3),
                                         optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += perf_counter()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp_old
Nelder-Mead     Time= 1.8030s  Loss=29155.95 [ 1030.81     0.   13758.24] Optimization terminated successfully.
L-BFGS-B        Time= 0.0527s  Loss=29162.94 [1030.83    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_old
Nelder-Mead     Time= 6.8086s  Loss=27751.59 [1.02e+03 0.00e+00 5.00e-01 3.36e+02 1.03e+02] Optimization terminated successfully.
L-BFGS-B        Time= 1.3027s  Loss=27756.39 [1.02e+03 0.00e+00 5.00e-01 3.37e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_old
Nelder-Mead     Time=386.8010s  Loss=21482.03 [1.72e+01 1.23e+02 2.14e+03 7.98e+03 9.93e-01 6.72e+03 1.34e+02 1.74e+01 1.73e+03] Optimization terminated successfully.
L-BFGS-B        Time=60.9116s  Loss=21603.43 [1.56e+00 3.82e+03 3.46e+04 2.22e+03 9.79e-01 3.82e+03 1.56e+02 1.88e+01 5.84e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### new parametrization w/ additive exps w/o multiplicative parameter interactions

In [11]:
single_exp_init = [trace[-1000:].mean(), 350, 3600]
double_exp_init = [trace[-1000:].mean(), 350, 200, 3600, 240]
bright_init = [trace[-1000:].mean(), 350, 200, 10, 10, 3600, 240, 50, 2000]

for model, start_params, bounds in (
    (single_exp, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_init, double_exp_bounds),
    (bright, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for optimizer in ("Nelder-Mead", "L-BFGS-B"):
        tic = -perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = nonlinear_fit(trace, timestamps, model, start_params, bounds=bounds,
                                         M=TukeyBiweight(3),
                                         optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += perf_counter()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp
Nelder-Mead     Time= 1.9133s  Loss=29155.95 [ 1030.81     0.   13055.65] Optimization terminated successfully.
L-BFGS-B        Time= 0.0326s  Loss=29162.94 [1030.83    0.   3252.99] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL

double_exp
Nelder-Mead     Time= 5.2738s  Loss=27751.59 [1.02e+03 1.30e-12 5.09e+02 3.40e+02 1.03e+02] Optimization terminated successfully.
L-BFGS-B        Time= 0.3895s  Loss=27756.39 [1017.82    0.    509.07 3190.88  102.92] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
Nelder-Mead     Time=508.5784s  Loss=21739.51 [8.27e+02 1.15e+06 1.96e+06 0.00e+00 3.10e+06 9.88e+02 9.30e+02 1.80e+02 9.51e+02] Maximum number of iterations has been exceeded.
L-BFGS-B        Time=49.0131s  Loss=21885.39 [8.23e+02 1.08e+05 3.63e+05 0.00e+00 4.69e+05 1.07e+03 8.91e+02 1.80e+02 9.30e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


### JAX autograd

#### old parametrization w/ multiplicative parameter interactions

In [12]:
single_exp_init = [trace[-1000:].mean(), 0.35, 3600]
double_exp_init = [trace[-1000:].mean(), 0.35, 0.2, 3600, 240]
bright_init = [trace[-1000:].mean(), 0.35, 0.2, 0.1, 0.1, 3600, 240, 50, 2000]

for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp_old, single_exp_init, single_exp_bounds),
        (double_exp_old, double_exp_init, double_exp_bounds),
        (bright_old, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -perf_counter()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = nonlinear_fit(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                        M=TukeyBiweight(3),
                                        backend="jax",
                                        dtype=dtype)
            tic += perf_counter()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)



float64

single_exp_old
L-BFGS-B        Time= 1.7280s  Loss=29162.92 [1030.83    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_old
L-BFGS-B        Time= 1.7845s  Loss=27756.36 [1.02e+03 0.00e+00 5.00e-01 3.37e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_old
L-BFGS-B        Time=18.9459s  Loss=21818.49 [1.77e+01 1.96e+02 1.42e+03 3.33e+02 9.62e-01 4.71e+03 1.58e+02 1.46e+02 3.31e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp_old
L-BFGS-B        Time= 1.8052s  Loss=29163.45 [1030.83    0.   3485.95] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_old
L-BFGS-B        Time= 1.2020s  Loss=27764.05 [1.02e+03 0.00e+00 5.12e-01 3.37e+03 9.71e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_old
L-BFGS-B        Time= 3.3359s  Loss=22835.26 [1.06e+03 5.70e-01 4.15e+00 0.00e+00 7.43e-01 3.60e+03 1.87e+02 5.39e+01 2.00e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR

#### new parametrization w/ additive exps w/o multiplicative parameter interactions

In [13]:
single_exp_init = [trace[-1000:].mean(), 350, 3600]
double_exp_init = [trace[-1000:].mean(), 350, 200, 3600, 240]
bright_init = [trace[-1000:].mean(), 350, 200, 10, 10, 3600, 240, 50, 2000]

for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp, single_exp_init, single_exp_bounds),
        (double_exp, double_exp_init, double_exp_bounds),
        (bright, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -perf_counter()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = nonlinear_fit(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                        M=TukeyBiweight(3),
                                        backend="jax",
                                        dtype=dtype)
            tic += perf_counter()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)



float64

single_exp
L-BFGS-B        Time= 0.6332s  Loss=29162.92 [1030.83    0.   3252.99] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL

double_exp
L-BFGS-B        Time= 1.0594s  Loss=27756.36 [1017.82    0.    509.07 3190.88  102.92] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
L-BFGS-B        Time=21.2050s  Loss=21888.79 [8.23e+02 9.59e+04 3.30e+05 0.00e+00 4.25e+05 1.07e+03 8.88e+02 1.80e+02 9.28e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp
L-BFGS-B        Time= 0.7528s  Loss=29163.45 [1030.83    0.   3252.99] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
L-BFGS-B        Time= 0.9572s  Loss=27756.37 [1017.84    0.    508.77 3190.86  102.83] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
L-BFGS-B        Time= 2.4941s  Loss=21739.95 [ 424.88 2215.28 3949.4     0.   5072.02 3607.56  528.1   178.71 1011.42] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


#### new parametrization and normalized trace (mean=1)

In [14]:
single_exp_init = [trace[-1000:].mean(), 0.35, 3600]
double_exp_init = [trace[-1000:].mean(), 0.35, 0.2, 3600, 240]
bright_init = [trace[-1000:].mean(), 0.35, 0.2, 0.1, 0.1, 3600, 240, 50, 2000]

tm = trace.mean()
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp, single_exp_init, single_exp_bounds),
        (double_exp, double_exp_init, double_exp_bounds),
        (bright, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -perf_counter()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = nonlinear_fit(trace/tm, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                        M=TukeyBiweight(3),
                                        backend="jax",
                                        dtype=dtype)
            tic += perf_counter()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2 * tm**2:.2f}", res.x, res.message)



float64

single_exp
L-BFGS-B        Time= 0.5015s  Loss=29299.39 [  0.98   0.   690.44] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL

double_exp
L-BFGS-B        Time= 1.3852s  Loss=27756.37 [9.70e-01 0.00e+00 4.85e-01 7.42e+02 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
L-BFGS-B        Time=92.3987s  Loss=21738.52 [7.59e-01 9.22e+00 3.10e+04 0.00e+00 3.11e+04 1.40e+03 8.27e+02 1.80e+02 8.27e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp
L-BFGS-B        Time= 0.6037s  Loss=29299.39 [  0.98   0.   690.44] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
L-BFGS-B        Time= 0.9359s  Loss=27947.08 [9.71e-01 0.00e+00 4.97e-01 7.41e+02 9.70e+01] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
L-BFGS-B        Time= 1.4512s  Loss=22123.40 [8.66e-02 5.88e+00 1.45e+00 0.00e+00 5.89e+00 3.60e+03 2.84e+02 5.60e+01 2.26e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
